In [1]:
# NOTEBOOK 27 - Round 1: can the wind-aware graph be made to win?  (Delhi v3 pipeline, ERA5 wind, 5 seeds)
# Two ideas, both selected on the VALIDATION set and scored on test:
#   B  context dropout : during training the three city-aggregate features (city mean / max / station-minus-city) are hidden
#                        for half of the windows, so the model cannot get all of its spatial information from them and
#                        has to use the graph. Applied to dynamic, static AND no-graph, so the comparison stays fair.
#   C  blend           : the graph used is  alpha * wind graph + (1 - alpha) * static graph, with alpha a learned function
#                        of the wind speed. It can fall back to the static graph, so it should never be worse than static.
# Existing checkpoints (dynamic / static / no-graph, no tricks) are the baselines. New checkpoints are saved as
# watgcn_v4<variant>_seed*.pt. Nothing existing is overwritten. About 5 minutes per model, 20 models, roughly 100 minutes.
# HOW TO CHECK: this cell only prints a note.
print('Round 1 variants: blend, dynamic_cd, static_cd, none_cd   (seeds 42/7/123/2024/999)')

Round 1 variants: blend, dynamic_cd, static_cd, none_cd   (seeds 42/7/123/2024/999)


In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [3]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')

T=19921  dyn_adj(ERA5) (19921, 37, 37)  static edges 508


In [4]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_30148\1524811802.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month


F=26  NaNs 0


In [5]:
feat_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
log_scaler  = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
Xs_full = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(T, N, F).astype(np.float32)
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Yl_full = to_log(Y_raw)

train_end_idx = int(T * 0.70); val_end_idx = int(T * 0.85)
def split_starts(lo, hi):
    return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
train_starts = split_starts(0, train_end_idx)
val_starts   = split_starts(train_end_idx, val_end_idx)
test_starts  = split_starts(val_end_idx, T)
del X_raw
print(f'windows -- train {len(train_starts):,} | val {len(val_starts):,} | test {len(test_starts):,}')

windows -- train 13,873 | val 2,917 | test 2,918


C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
class DS(Dataset):
    def __init__(self, starts, weights=None):
        self.s = np.asarray(starts, np.int64)
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)
    def __len__(self):
        return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(Xs_full[g:i1].copy()),
                torch.from_numpy(dyn_adj[g:i1].copy()),
                torch.from_numpy(dyn_adj[i1:i2].copy()),
                torch.from_numpy(np.ascontiguousarray(Xs_full[i1:i2][:, :, FUT_IDX])),
                torch.from_numpy(Yl_full[i1-1].copy()),
                torch.from_numpy(Yl_full[i1:i2].copy()),
                torch.tensor(self.w[k], dtype=torch.float32))

def window_swing(starts):
    return np.array([(Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].max(0) -
                      Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].min(0)).mean()
                     for g in starts], dtype=np.float32)
tr_sw = window_swing(train_starts); med = float(np.median(tr_sw))
tr_w = np.clip(1.0 + LOSS_VOL_ALPHA * (tr_sw / (med + 1e-6)), 0.5, 4.0).astype(np.float32)
tr_w *= len(tr_w) / tr_w.sum()

tr_ld = DataLoader(DS(train_starts, tr_w), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
va_ld = DataLoader(DS(val_starts),         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
te_ld = DataLoader(DS(test_starts),        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'batches -- train {len(tr_ld)} | val {len(va_ld)} | test {len(te_ld)}')

batches -- train 868 | val 183 | test 183


In [7]:
# model with three graph modes plus the learned blend, and optional context dropout at training time
CTX_COLS = ['city_mean_pm', 'city_max_pm', 'pm_minus_city']
CTX_IDX = [FEATURE_COLS.index(c) for c in CTX_COLS]
WSPD_IDX = FEATURE_COLS.index('wspd')
FUT_WSPD = 9                                  # wspd is the last of the 10 future channels (7 calendar + u, v, wspd)

class WATGCNv4(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon, n_stations,
                 dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        sa = sa.clone(); sa.fill_diagonal_(0)
        self.register_buffer('static_adj', sa, persistent=False)
        self.register_buffer('static_rn', sa / sa.sum(1, keepdim=True).clamp(min=1.0), persistent=False)   # row-normalised
        if graph_mode == 'blend':
            self.blend_a = nn.Parameter(torch.zeros(1)); self.blend_b = nn.Parameter(torch.zeros(1))

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def alpha(self, speed):                       # speed: (B, steps) standardised wind speed -> weight of the wind graph
        return torch.sigmoid(self.blend_a + self.blend_b * speed)[..., None, None]

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            A = None
        elif self.graph_mode == 'dynamic':
            A = self._n(At + I.view(1, 1, n, n))
        elif self.graph_mode == 'static':
            A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
        else:                                      # blend
            a = self.alpha(X[..., WSPD_IDX].mean(-1))
            A = self._n(a * At + (1 - a) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(h if A is None else torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        elif self.graph_mode == 'static':
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        else:
            af = self.alpha(Fut[..., FUT_WSPD].mean(-1))
            ctx_fc = torch.matmul(self._n(af * Afc + (1 - af) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n)), ctx_rep)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, self.fut_enc(Fut)], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

STATIC_NP = np.load(STATIC_ADJ_NPY).astype(np.float32)
VARIANTS = {            # name -> (graph mode, context-dropout probability)
    'blend':      ('blend', 0.0),
    'dynamic_cd': ('dynamic', 0.5),
    'static_cd':  ('static', 0.5),
    'none_cd':    ('none', 0.5),
}
def build_v(name, n=None, static_adj=None):
    mode, _ = VARIANTS[name] if name in VARIANTS else (name, 0.0)
    return WATGCNv4(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, n or N, DROPOUT, N_FUT, GCN_HOPS, graph_mode=mode,
                    static_adj=STATIC_NP if static_adj is None else static_adj).to(DEVICE)
for v in VARIANTS:
    _m = build_v(v); print(f'{v:11s} {sum(p.numel() for p in _m.parameters()):,} params'); del _m

# HOW TO CHECK: 18,297 params for static/dynamic/none variants, 18,299 for blend (two extra scalars).

blend       18,299 params
dynamic_cd  18,297 params
static_cd   18,297 params
none_cd     18,297 params


In [8]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

In [9]:
# training with context dropout (only while training; evaluation always sees the real features)
def run_epoch_v(model, loader, opt=None, sched=None, cd_p=0.0):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        if tr and cd_p > 0:
            hide = (torch.rand(X.size(0), 1, 1, 1, device=DEVICE) < cd_p).float()          # one coin per window
            X = X.clone(); X[..., CTX_IDX] = X[..., CTX_IDX] * (1 - hide)                   # 0 = training mean after scaling
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

_prog = {'done': 0, 'total': 0, 't0': time.time(), 'times': []}
def _progress(name, seed, ep, vl, best):
    el = (time.time() - _prog['t0']) / 60
    left = (np.mean(_prog['times']) * (_prog['total'] - _prog['done'] - 1)) if _prog['times'] else None
    line = (f"[model {_prog['done'] + 1}/{_prog['total']}] {name} seed {seed} | epoch {ep} | val {vl:.4f} (best {best:.4f}) | "
            f"elapsed {el:.0f} min" + (f" | ~{left:.0f} min left after this model" if left is not None else ''))
    print(line + ' ' * 12, end='\r', flush=True)


def train_variant(name, seed, max_epochs=60, patience=12, lr=1e-3):
    mode, cd_p = VARIANTS[name]
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_v(name)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(tr_ld) * max_epochs; warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/warm if s < warm else 0.5*(1 + np.cos(np.pi*(s-warm)/max(1, steps-warm))))
    best, best_state, ctr = 1e9, None, 0
    for ep in range(1, max_epochs + 1):
        t0 = time.time()
        trl = run_epoch_v(model, tr_ld, opt, sched, cd_p); vl = run_epoch_v(model, va_ld)
        if vl < best - 1e-5: best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
        else: ctr += 1
        _progress(name, seed, ep, vl, best)
        if ctr >= patience: break
    model.load_state_dict(best_state)
    return model
print('ready')

ready


In [10]:
# train and save the new variants (resumes: a seed whose checkpoint exists is loaded, not retrained)
def ckpt_v(name, sd):
    base = {'dynamic': f'watgcn_v3_seed{sd}.pt', 'static': f'watgcn_v3static_seed{sd}.pt', 'none': f'watgcn_v3none_seed{sd}.pt'}
    return os.path.join(MODEL_DIR, base[name] if name in base else f'watgcn_v4{name}_seed{sd}.pt')

ALL = ['none', 'static', 'dynamic'] + list(VARIANTS)
_todo = [(n_, sd_) for n_ in ALL for sd_ in SEEDS if not os.path.exists(ckpt_v(n_, sd_))]
_prog['total'] = len(_todo); _prog['t0'] = time.time()
print(f'{len(_todo)} models to train, {len(ALL) * len(SEEDS) - len(_todo)} already saved')
pred = {'val': {}, 'test': {}}
for name in ALL:
    for split, ld in (('val', va_ld), ('test', te_ld)):
        pred[split][name] = []
    for sd in SEEDS:
        f = ckpt_v(name, sd)
        if os.path.exists(f):
            m = build_v(name); m.load_state_dict(torch.load(f, map_location=DEVICE)); tag = 'loaded'
        else:
            _t = time.time(); m = train_variant(name, sd); torch.save(m.state_dict(), f); tag = 'trained'
            _prog['times'].append((time.time() - _t) / 60); _prog['done'] += 1; print()
        for split, ld in (('val', va_ld), ('test', te_ld)):
            p, a = predict(m, ld); pred[split][name].append(p)
            if split == 'val': val_actual = a
            else: test_actual = a
        print(f'[{name}] seed {sd} {tag}')
        del m; torch.cuda.empty_cache()

# HOW TO CHECK: about 100 minutes the first time. If it is interrupted, run this cell again: finished seeds are loaded.

15 models to train, 20 already saved
[none] seed 42 loaded
[none] seed 7 loaded
[none] seed 123 loaded
[none] seed 2024 loaded
[none] seed 999 loaded
[static] seed 42 loaded
[static] seed 7 loaded
[static] seed 123 loaded
[static] seed 2024 loaded
[static] seed 999 loaded
[dynamic] seed 42 loaded
[dynamic] seed 7 loaded
[dynamic] seed 123 loaded
[dynamic] seed 2024 loaded
[dynamic] seed 999 loaded
[blend] seed 42 loaded
[blend] seed 7 loaded
[blend] seed 123 loaded
[blend] seed 2024 loaded
[blend] seed 999 loaded
[model 1/15] dynamic_cd seed 42 | epoch 16 | val 0.1556 (best 0.1400) | elapsed 7 min            
[dynamic_cd] seed 42 trained
[model 2/15] dynamic_cd seed 7 | epoch 14 | val 0.1570 (best 0.1386) | elapsed 12 min | ~74 min left after this model            
[dynamic_cd] seed 7 trained
[model 3/15] dynamic_cd seed 123 | epoch 17 | val 0.1571 (best 0.1355) | elapsed 17 min | ~62 min left after this model            
[dynamic_cd] seed 123 trained
[model 4/15] dynamic_cd seed 2024 

In [11]:
# Delhi results: validation (used to choose) and test (scored once)
def ens_scores(split, act):
    return {n: sc(np.mean(pred[split][n], axis=0), act) for n in ALL}
SV, ST = ens_scores('val', val_actual), ens_scores('test', test_actual)
print(f"{'variant':14}{'val RMSE':>10}{'test RMSE':>11}{'test MAE':>10}{'test R2':>9}")
for n in ALL:
    print(f"{n:14}{SV[n]['rmse']:>10.2f}{ST[n]['rmse']:>11.2f}{ST[n]['mae']:>10.2f}{ST[n]['r2']:>9.3f}")

# choose on VALIDATION: best wind-graph variant vs best non-wind variant
WIND = ['dynamic', 'dynamic_cd', 'blend']; OTHER = ['none', 'static', 'none_cd', 'static_cd']
bw = min(WIND, key=lambda n: SV[n]['rmse']); bo = min(OTHER, key=lambda n: SV[n]['rmse'])
print(f"\nchosen on validation: wind-graph = {bw} (val {SV[bw]['rmse']:.2f}) | best non-wind = {bo} (val {SV[bo]['rmse']:.2f})")
print(f"TEST: {bw} {ST[bw]['rmse']:.2f} vs {bo} {ST[bo]['rmse']:.2f}   -> wind graph advantage {ST[bo]['rmse'] - ST[bw]['rmse']:+.2f} ug/m3  (target: at least +1.00)")
print('Seed noise note: single models vary by 1-2 ug/m3; this is a 5-seed ensemble comparison, so anything under ~1 is not conclusive.')
json.dump({'val': {n: SV[n] for n in ALL}, 'test': {n: ST[n] for n in ALL}, 'chosen_wind': bw, 'chosen_other': bo,
           'advantage_test': ST[bo]['rmse'] - ST[bw]['rmse']}, open(os.path.join(RESULTS_DIR, 'v4_round1_delhi.json'), 'w'), indent=2)

# by forecast-wind regime for the chosen pair (does the wind graph win when transport is active?)
ar = from_log(test_actual)
era5_spd_all = np.hypot(era5[..., 0], era5[..., 1])
fut_spd = np.array([era5_spd_all[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].mean() for g in np.arange(int(len(era5)*0.85), len(era5)-(INPUT_WINDOW+HORIZON)+1)])
w33, w66 = np.percentile(fut_spd, [33, 66])
regs = {'calm': fut_spd <= w33, 'breezy': (fut_spd > w33) & (fut_spd <= w66), 'windy': fut_spd > w66}
print(f"\n{'regime':8}{bw:>14}{bo:>14}{'advantage':>11}")
for k, msk in regs.items():
    a_ = np.sqrt(np.mean((from_log(np.mean(pred['test'][bw], 0))[msk] - ar[msk]) ** 2))
    b_ = np.sqrt(np.mean((from_log(np.mean(pred['test'][bo], 0))[msk] - ar[msk]) ** 2))
    print(f'{k:8}{a_:>14.2f}{b_:>14.2f}{b_ - a_:>+11.2f}')

variant         val RMSE  test RMSE  test MAE  test R2
none               56.03      58.40     37.78    0.664
static             55.40      58.23     38.15    0.666
dynamic            55.81      59.08     38.27    0.656
blend              56.52      59.00     38.17    0.657
dynamic_cd         56.38      58.81     38.03    0.659
static_cd          55.99      58.17     38.00    0.666
none_cd            55.75      58.24     37.95    0.666

chosen on validation: wind-graph = dynamic (val 55.81) | best non-wind = static (val 55.40)
TEST: dynamic 59.08 vs static 58.23   -> wind graph advantage -0.85 ug/m3  (target: at least +1.00)
Seed noise note: single models vary by 1-2 ug/m3; this is a 5-seed ensemble comparison, so anything under ~1 is not conclusive.

regime         dynamic        static  advantage
calm             80.48         78.58      -1.90
breezy           49.61         49.07      -0.53
windy            39.89         40.53      +0.64


In [12]:
# cross-city, zero-shot: the same variants on Kolkata / Mumbai / Ahmedabad / Patna (data from notebook 24)
import pickle
WORK = os.path.join(V2_DIR, 'crosscity_cache')
CITIES = ['Kolkata', 'Mumbai', 'Ahmedabad', 'Patna']
TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
class City:
    def __init__(self, name):
        z = np.load(os.path.join(WORK, f'city_{name}.npz'))
        self.name = name; self.X = z['X_raw']; self.dyn = z['dyn_adj']; self.static = z['static_adj']; self.obs = z['obs_mask']
        self.grid = pd.DatetimeIndex(z['grid']); self.T, self.n, _ = self.X.shape
        self.Xs = feat_scaler.transform(self.X.reshape(-1, F)).reshape(self.T, self.n, F).astype(np.float32)
        self.Yl = to_log(self.X[:, :, 0])
        t_test = int(np.searchsorted(self.grid, pd.Timestamp('2025-12-08')))
        self.test_starts = np.arange(t_test, self.T - (INPUT_WINDOW + HORIZON) + 1)
class CityDS(Dataset):
    def __init__(self, c, starts): self.c, self.s = c, np.asarray(starts, np.int64)
    def __len__(self): return len(self.s)
    def __getitem__(self, k):
        c = self.c; g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(c.Xs[g:i1].copy()), torch.from_numpy(c.dyn[g:i1].copy()), torch.from_numpy(c.dyn[i1:i2].copy()),
                torch.from_numpy(np.ascontiguousarray(c.Xs[i1:i2][:, :, FUT_IDX])), torch.from_numpy(c.Yl[i1-1].copy()),
                torch.from_numpy(c.Yl[i1:i2].copy()), torch.tensor(1.0))

cc = {}
for cname in CITIES:
    c = City(cname); ld = DataLoader(CityDS(c, c.test_starts), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    mask = np.stack([c.obs[g + INPUT_WINDOW: g + INPUT_WINDOW + HORIZON] for g in c.test_starts])
    anc = np.stack([c.Yl[g + INPUT_WINDOW - 1] for g in c.test_starts])
    r = {}
    for name in ALL:
        ps = []
        for sd in SEEDS:
            m = build_v(name, c.n, c.static); m.load_state_dict(torch.load(ckpt_v(name, sd), map_location=DEVICE))
            p, a = predict(m, ld); ps.append(p); del m
        pr = from_log(np.mean(ps, 0)); ac = from_log(a)
        r[name] = float(np.sqrt(np.mean((pr[mask] - ac[mask]) ** 2)))
    pers = from_log(np.repeat(anc[:, None, :], HORIZON, axis=1)); r['persistence'] = float(np.sqrt(np.mean((pers[mask] - ac[mask]) ** 2)))
    cc[cname] = r
    print(f'{cname}: done')
print()
print(f"{'variant':14}" + ''.join(f'{c:>11}' for c in CITIES) + f"{'mean':>9}")
for name in ['persistence'] + ALL:
    vals = [cc[c][name] for c in CITIES]
    print(f'{name:14}' + ''.join(f'{v:>11.2f}' for v in vals) + f'{np.mean(vals):>9.2f}')
bo_c = {c: min(cc[c][n] for n in OTHER + ['none', 'static']) for c in CITIES}
print(f"\nchosen wind-graph variant {bw}: advantage over the best non-wind variant per city (positive = wind graph better):")
print('  ' + ' | '.join(f"{c} {bo_c[c] - cc[c][bw]:+.2f}" for c in CITIES))
json.dump(cc, open(os.path.join(RESULTS_DIR, 'v4_round1_crosscity.json'), 'w'), indent=2)

# HOW TO READ: the aim is a wind-graph variant that is better than every non-wind variant by about 1 ug/m3 on Delhi
# and not worse in at least 3 of the 4 cities. If it is not, Round 2 (transport-aligned features) is next.

Kolkata: done
Mumbai: done
Ahmedabad: done
Patna: done

variant           Kolkata     Mumbai  Ahmedabad      Patna     mean
persistence         44.60      33.13      35.67      39.25    38.16
none                30.46      29.00      27.04      32.83    29.83
static              31.22      28.90      29.16      32.37    30.41
dynamic             31.35      29.02      27.47      32.38    30.06
blend               31.04      28.71      27.24      32.46    29.86
dynamic_cd          31.17      28.59      27.40      32.00    29.79
static_cd           31.72      28.52      29.20      32.31    30.44
none_cd             30.64      28.54      26.87      32.87    29.73

chosen wind-graph variant dynamic: advantage over the best non-wind variant per city (positive = wind graph better):
  Kolkata -0.89 | Mumbai -0.50 | Ahmedabad -0.60 | Patna -0.07
